In [3]:
import pandas as pd 
df = pd.read_excel('/Users/egorilin/Desktop/COMA/2026_04_14_DataSet_Final.xlsx')
smiles = df['SMILES']
smiles

0                                     OCC[N+](C)(C)C.[Br-]
1                     C[N+](C)(C)CCOC(CCCCCCCCCCC)=O.[Br-]
2                                  OCC[N+](CC)(CC)CC.[Br-]
3                  CC[N+](CC)(CCOC(CCCCCCCCCCC)=O)CC.[Br-]
4              CC[N+](CC)(CCOC(CCCCCCCCCCCCCCC)=O)CC.[Br-]
                               ...                        
13399    O=C1N(CCCCCCCCCC)C2=CC=CC=C2/C1=N/NC(C[N+](C)(...
13400    O=C1N(CCCCCCCCCCCC)C2=CC=CC=C2/C1=N/NC(C[N+](C...
13401    O=C1N(CCCCCCCCCCCCCC)C2=CC=CC=C2/C1=N/NC(C[N+]...
13402              CCCCCCCCCCCCCCCC[N+]1(CC2)CCN2CC1.[Br-]
13403              CCCCCCCCCCCCCCCC[N+]1(CC2)CCN2CC1.[Br-]
Name: SMILES, Length: 13404, dtype: object

In [9]:
import pandas as pd
import numpy as np
import re
import os
import warnings

# Игнорируем предупреждения pandas 2.x о deprecated behavior в groupby
warnings.filterwarnings('ignore', category=FutureWarning)

INPUT_FILE = '2026_04_14_DataSet_Final.xlsx'
OUTPUT_FILE = '2026_04_14_DataSet_Final_Merged.xlsx'

def parse_mic_value(val):
    """Безопасно преобразует строки MIC/MBC в float."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip().replace(',', '.')
    s_clean = re.sub(r'[<>=≤≥≦≧~]', '', s)
    s_clean = re.sub(r'[^\d.\-]', '', s_clean)
    try:
        return float(s_clean) if s_clean else np.nan
    except ValueError:
        return np.nan

def main():
    if not os.path.exists(INPUT_FILE):
        raise FileNotFoundError(f"Файл {INPUT_FILE} не найден.")

    print("⏳ Загрузка данных...")
    df = pd.read_excel(INPUT_FILE, engine='openpyxl')
    df.columns = df.columns.str.strip()
    print(f"📊 Исходный размер: {df.shape[0]} строк, {df.shape[1]} столбцов")

    KEY_COLS = ['Canonical SMILES', 'Strain']
    missing = [c for c in KEY_COLS if c not in df.columns]
    if missing:
        raise KeyError(f"Не найдены ключевые столбцы: {missing}")

    # Очистка ключей
    for c in KEY_COLS:
        df[c] = df[c].astype(str).str.strip()
    df = df.dropna(subset=KEY_COLS)

    # Парсинг числовых MIC/MBC
    TARGET_COLS = ['MIC, µM', 'MBC, µM', 'MIC, mg/L', 'MBC, mg/L']
    num_cols_clean = []
    for col in TARGET_COLS:
        if col in df.columns:
            num_col = f'{col}_num'
            df[num_col] = df[col].apply(parse_mic_value)
            num_cols_clean.append(num_col)

    print("🔍 Группировка и агрегация...")
    grouped = df.groupby(KEY_COLS, sort=False)
    results = {}

    # 1️⃣ Числовые метрики (встроенные методы работают безотказно)
    for col in num_cols_clean:
        results[f'{col}_min'] = grouped[col].min()
        results[f'{col}_median'] = grouped[col].median()
        results[f'{col}_count'] = grouped[col].count()

    # 2️⃣ Метаданные (склейка уникальных источников)
    meta_cols = ['Name', 'CODE', 'doi', 'Reference (First author, Year, Volume, Issue, Pages)', 'Year']
    for col in meta_cols:
        if col in df.columns:
            # .agg() гарантирует возврат скаляра и не ломается на индексах
            results[col] = grouped[col].agg(
                lambda s: ' | '.join(s.dropna().astype(str).unique())
            )

    # 3️⃣ Остальные колонки (первое непустое значение)
    other_cols = [c for c in df.columns if c not in results and c not in KEY_COLS and not c.endswith('_num')]
    for col in other_cols:
        if col in df.columns:
            results[col] = grouped[col].agg(
                lambda s: s.dropna().iloc[0] if len(s.dropna()) > 0 else np.nan
            )

    # Безопасная сборка финального DataFrame
    df_merged = pd.DataFrame(results).reset_index()

    # Сортировка по минимальному MIC (самые активные сверху)
    if 'MIC, µM_num_min' in df_merged.columns:
        df_merged = df_merged.sort_values('MIC, µM_num_min', na_position='last').reset_index(drop=True)

    # Сохранение
    df_merged.to_excel(OUTPUT_FILE, index=False, engine='openpyxl')
    print(f"✅ Готово! Результат сохранен в: {OUTPUT_FILE}")
    print(f"📉 Итоговый размер: {df_merged.shape[0]} строк (объединено: {df.shape[0] - df_merged.shape[0]} записей)")

if __name__ == '__main__':
    main()

⏳ Загрузка данных...
📊 Исходный размер: 13404 строк, 38 столбцов
🔍 Группировка и агрегация...


ValueError: 2